## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. Just press ▶ on the cell below and wait
for the green **✅ Setup complete**, then run the rest top to bottom.

When it asks to **connect Google Drive**, click **Connect** — that lets the data
file download **only once** (it's saved to your Drive and reused by every
notebook) and saves your figures for your poster. You *can* skip it, but then each
notebook re-downloads the ~470 MB data and your figures won't be saved.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os

print("1/3  installing libraries ...")
get_ipython().system('pip install -q "mne==1.10.1" gdown')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

# Connect Drive so the data is downloaded ONCE (saved to your Drive) and your
# figures persist. If you skip it, we fall back to temporary storage.
print("3/3  connecting Google Drive ...")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = "/content/drive/MyDrive/DecodingBrain_data"
    os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
    saved = True
except Exception:
    data_dir = "data"                     # temporary (re-downloads each session)
    os.environ["CAMP_OUTPUT_DIR"] = "outputs"
    saved = False

os.makedirs(data_dir, exist_ok=True)
data_path = os.path.join(data_dir, "synapse_preprocessed.pkl")
os.environ["CAMP_DATA_PATH"] = data_path

if os.path.exists(data_path):
    print("     data already saved in your Drive — skipping download \u26a1")
else:
    print("     downloading the data (~470 MB, one time only) ...")
    import gdown
    gdown.download(id="1Z-NENlKMjL-kL-N46lQ8QA1AbGM7bJHY", output=data_path, quiet=False)

print("\n\u2705 Setup complete.",
      "Data + figures are saved in your Drive (DecodingBrain_*)." if saved
      else "Heads up: you skipped Drive, so the data re-downloads each session.")


# Week 2 · Day 6 — Feature Extraction

So far each calculation gave us one number for one subject. To do real data
science we need a **table**: one **row per subject**, one **column per feature**.
This "subjects × features" matrix is the foundation for *everything* in Weeks 2
and 3 — statistics, correlations, and machine learning all read from it.

### By the end of this notebook you will be able to
1. Define what a "feature" is and name features systematically
2. Build a tidy `pandas` DataFrame of PSD + ERP features
3. Add a `group` label column
4. Save the feature table for later notebooks to reuse

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import camp_utils as cu

data = cu.load_camp_data(verbose=False)

## 1. What is a feature?
A **feature** is one measurable number that describes a recording. We've already
made two kinds:
- **PSD features:** band-power change in dB (e.g. "LET gamma")
- **ERP features:** component amplitude or latency (e.g. "LET N1 amplitude")

We give each feature a clear name like `let_gamma` or `let_n1_amp` so we never
lose track of what a column means.

## 2. One subject's feature dictionary
Let's compute a handful of features for a single subject and store them in a
dictionary `{feature_name: value}`.

In [ ]:
def extract_features_for_subject(epochs_by_task):
    """Compute a dictionary of features for ONE subject.

    epochs_by_task: dict like {"let": <Epochs or None>, "hlt": ..., ...}
    """
    feats = {}

    # --- PSD features: every band, for the auditory tasks ---
    for task in ["let", "hlt", "ast"]:
        ep = epochs_by_task.get(task)
        if ep is None:
            continue
        window = cu.get_time_windows(task)["full_stim"]
        for band in cu.BAND_ORDER:
            feats[f"{task}_{band}"] = cu.band_power_db(ep, band, window)

    # --- ERP features: N1 and P2 amplitude for the LET task ---
    let_ep = epochs_by_task.get("let")
    if let_ep is not None:
        for comp in ["n1", "p2"]:
            amp, lat = cu.erp_amplitude_and_latency(let_ep, comp)
            feats[f"let_{comp}_amp"] = amp
            feats[f"let_{comp}_lat"] = lat

    return feats


# test on one subject
example = {task: data["exp_epochs"][task][0] for task in cu.TASKS}
example_feats = extract_features_for_subject(example)
print(f"Computed {len(example_feats)} features for {data['exp_subjects'][0]}:")
for name, val in list(example_feats.items())[:6]:
    print(f"   {name:14s} = {val:+.2f}")
print("   ...")

## 3. Build the table for everyone
Now we loop over **all** subjects in **both** groups, collecting one dictionary
per subject, and let `pandas` turn the list of dicts into a DataFrame.

### ✏️ Your turn #1 — finish the loop
The outer loop and the row collection are set up. You add the `group` label and
the `subject` id to each row.

In [ ]:
rows = []
for group in ["exp", "ctrl"]:
    subjects = data[f"{group}_subjects"]
    for i, subject in enumerate(subjects):
        # gather this subject's epochs for every task (some may be None)
        epochs_by_task = {task: data[f"{group}_epochs"][task][i] for task in cu.TASKS}
        feats = extract_features_for_subject(epochs_by_task)

        # TODO: add two more entries to the feats dict:
        #   feats["subject"] = ...   (the subject id)
        #   feats["group"]   = ...   ("EXP" or "CTRL" — uppercase)
        # hint: group.upper() turns "exp" into "EXP"

        rows.append(feats)

features = pd.DataFrame(rows)
# put subject and group first for readability
id_cols = ["subject", "group"]
feature_cols = [c for c in features.columns if c not in id_cols]
features = features[id_cols + feature_cols]

print("Feature table shape:", features.shape, "(rows = subjects, cols = id + features)")
features.head()

In [ ]:
cu.check("group" in features.columns and "subject" in features.columns
         and set(features["group"].unique()) == {"EXP", "CTRL"},
         "Table has subject + group columns with EXP/CTRL labels.",
         "Add feats['subject'] and feats['group'] (uppercase) inside the loop.")

## 4. Inspect the table
A few sanity checks data scientists always do:

In [ ]:
print("How many subjects per group?")
print(features["group"].value_counts())
print("\nAny missing values per feature?")
print(features[feature_cols].isna().sum().sort_values(ascending=False).head())

Compare the group averages for one feature:

In [ ]:
print(features.groupby("group")["let_gamma"].agg(["mean", "std", "count"]))

### ✏️ Your turn #2 — add a feature
Extend `extract_features_for_subject` (scroll up and edit it) to also compute the
**LET P3 amplitude** (`let_p3_amp`). Re-run the table-building cell, then check:

In [ ]:
cu.check("let_p3_amp" in features.columns,
         "let_p3_amp is now in the table!",
         "Add 'p3' to the component loop in extract_features_for_subject, then "
         "re-run the build cell in section 3.")

## 5. Save it
Other notebooks (statistics, correlations, machine learning) will load this file
instead of recomputing everything. Saving intermediate results is good practice.

In [ ]:
out = cu.save_path("features_table.csv")
features.to_csv(out, index=False)
print("Saved feature table to", out)
print("From now on you can load it with:  pd.read_csv(cu.save_path('features_table.csv'))")

## 🎯 Wrap-up
You built the central object of the whole project: a subjects × features table.
Every analysis from here reads from a table like this.

**Think about it:** We computed ~21 features from 28 people. In machine learning,
having *more features than people* is risky — why might that be a problem? (We'll
tackle it head-on in Week 3.)

➡️ **Next:** Notebook 06 — making these comparisons look like a real paper figure.